# TD 5 · Généraliser : régulariser, et le prouver par une ablation

**Bloc 3 : deep learning · Séance 10 · Durée : 85 min · Niveau : ★★☆ à ★★★**

---

## Objectifs

À la fin de ce TD, vous serez capable de :

1. mettre en évidence l'overfitting et mesurer le gap train / validation ;
2. relier weight decay et Ridge, en écrivant la pénalité L2 à la main ;
3. implémenter le dropout (masque et remise à l'échelle) et vérifier son espérance ;
4. distinguer batch norm et layer norm par le calcul, y compris leur dépendance au batch ;
5. concevoir et exécuter une ablation contrôlée : un facteur à la fois, trois seeds,
   un seuil fixé **avant** de regarder ;
6. mesurer le coût d'un `model.eval()` oublié.

## Prérequis

- TD 3 et TD 4 ; `src/training.py` ;
- séance 4 (Ridge, bias-variance tradeoff), séance 6 (bruit d'une estimation, seuil fixé à
  l'avance) ;
- cours : partie 3 du diaporama (séance 10).

## Concepts mobilisés

train/validation gap · early stopping · weight decay (L2) · dropout · batch norm ·
layer norm · data augmentation (bruit gaussien) · ablation · seeds · seuil de
deux écarts-types · interactions entre régularisateurs · `model.eval()`.

## Déroulé

| Exercice | Contenu                                       | Durée  | Type                | Niveau |
| -------- | --------------------------------------------- | ------ | ------------------- | ------ |
| 5.1      | Le point de départ : un réseau en overfitting | 8 min  | guidé               | ★☆☆    |
| 5.2      | Weight decay = Ridge                          | 15 min | compléter, comparer | ★★☆    |
| 5.3      | Le dropout à la main                          | 12 min | compléter           | ★★☆    |
| 5.4      | Batch norm et layer norm                      | 12 min | compléter, analyser | ★★☆    |
| 5.5      | L'ablation                                    | 28 min | résoudre            | ★★★    |
| 5.6      | Le bug silencieux                             | 5 min  | analyser            | ★☆☆    |
|          | Synthèse et questions                         | 5 min  |                     |        |

## Résultats attendus

- Le réseau de 61 185 paramètres de la séance 10 : AUC 0,8019 et un gap final de
  0,961 pour la seed 0.
- Pénalité écrite à la main et option `weight_decay` de l'optimizer : mêmes poids.
- Un dropout qui conserve l'espérance, et qui disparaît en eval mode.
- Batch norm : un exemple ne donne pas la même sortie selon ses voisins de batch ; layer
  norm : si.
- Les six premières lignes de votre ablation **reproduisent exactement** le tableau de
  la séance 10 ; deux lignes nouvelles (layer norm, bruit) y ajoutent deux constats.
- Un `model.eval()` oublié coûte 0,0280 d'AUC, sans aucun message.

## Mode d'emploi

La cellule d'ablation (5.5) est la seule longue : environ une minute. Lancez-la
dès que vos prédictions sont écrites, puis travaillez sur 5.6 pendant qu'elle tourne.


## Mise en route


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import re

import torch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.data import SEED, load_raw, set_seed, split_by_host
from src.training import (
    make_mlp,
)

set_seed(SEED)
# Quatre threads : les conditions dans lesquelles la séance 10 a été exécutée (TD 3).
torch.set_num_threads(4)

df = load_raw()
train, _ = split_by_host(df)
train["licensed"] = train["license"].map(
    lambda s: bool(re.search(r"HUT|HB-|AJ0|ESFC", str(s))) if str(s) != "nan" else False
)
NUMERIC = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "latitude",
    "longitude",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
    "calculated_host_listings_count",
    "availability_365",
]
CATEGORICAL = ["room_type", "property_type", "neighbourhood_cleansed"]
X = train[NUMERIC + CATEGORICAL]
y = train["licensed"].astype(int)
train_idx, val_idx = list(GroupKFold(n_splits=5).split(X, y, groups=train["host_id"]))[
    0
]
prep = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            NUMERIC,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            min_frequency=20,
                            sparse_output=False,
                        ),
                    ),
                ]
            ),
            CATEGORICAL,
        ),
    ]
).fit(X.iloc[train_idx])
X_tr, X_va = prep.transform(X.iloc[train_idx]), prep.transform(X.iloc[val_idx])
y_tr, y_va = y.iloc[train_idx].to_numpy(), y.iloc[val_idx].to_numpy()
N_NUM = len(NUMERIC)  # les 11 premières colonnes sont numériques et standardisées
BIG = (256, 128, 64)
print(f"fold 0 : training set {X_tr.shape}, validation set {X_va.shape}")
print(
    f"architecture {BIG} : {sum(p.numel() for p in make_mlp(X_tr.shape[1], hidden=BIG).parameters()):,} "
    f"paramètres pour {len(X_tr):,} lignes d'entraînement"
)

---

## Exercice 5.1 · Le point de départ : un réseau en overfitting

_Guidé · 8 min · ★☆☆_

**Explication.** Pour étudier la régularisation, il faut un réseau qui fait vraiment
de l'overfitting : six fois plus de paramètres que de lignes. La quantité à surveiller
est le **gap** entre la validation loss et la training loss.

**Exemple.** La fonction d'entraînement de la séance 10, à l'identique. Elle servira
de base à tout le TD.


In [ ]:
# Le training set, dans l'ordre et sans mélange : pour mesurer la training loss du
# modèle final.
train_eval_loader = DataLoader(
    TabularDataset(X_tr, y_tr), batch_size=1024, shuffle=False
)


def train_once(
    seed,
    *,
    dropout=0.0,
    weight_decay=0.0,
    batch_norm=False,
    early_stop=False,
    epochs=40,
    hidden=BIG,
    batch_size=512,
    lr=1e-3,
):
    """Un entraînement de la séance 10 : (AUC, epochs, gap final, historique)."""
    tr_loader, va_loader = loaders_from_arrays(
        X_tr, y_tr, X_va, y_va, batch_size=batch_size, seed=seed
    )
    torch.manual_seed(seed)
    net = make_mlp(X_tr.shape[1], hidden=hidden, dropout=dropout, batch_norm=batch_norm)
    stopper = EarlyStopping(patience=5) if early_stop else None
    history = fit(
        net,
        tr_loader,
        va_loader,
        epochs=epochs,
        lr=lr,
        weight_decay=weight_decay,
        early_stopping=stopper,
        seed=seed,
    )
    val_loss, logits = evaluate(net, va_loader, nn.BCEWithLogitsLoss())
    # Le gap : les deux losses mesurées sur le modèle final (celui que l'early stopping a
    # restauré), en eval mode, donc sans dropout. Les losses de la dernière epoch
    # décriraient un autre modèle.
    train_loss, _ = evaluate(net, train_eval_loader, nn.BCEWithLogitsLoss())
    gap = val_loss - train_loss
    return roc_auc_score(y_va, logits.ravel()), len(history["train"]), gap, history


auc, n_ep, ecart, historique = train_once(0)
print(f"  sans régularisation : AUC {auc:.4f}   epochs {n_ep}   gap final {ecart:.3f}")
ok = abs(auc - 0.8019) < 5e-4 and abs(ecart - 0.961) < 5e-3
print(
    f"  identique à la séance 10 : {'OK' if ok else 'différent (vérifiez le nombre de threads)'}"
)

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(historique["train"], label="training loss")
ax.plot(historique["val"], label="validation loss")
ax.set(
    xlabel="epoch",
    ylabel="cross-entropy",
    title="61 185 paramètres pour 10 100 lignes : le gap se creuse",
)
ax.legend()
plt.tight_layout()
plt.show()

**Analyse.** La training loss tend vers zéro, la validation loss remonte après
quelques epochs : le gap final, mesuré sur le modèle final en eval mode,
dépasse 0,9. Le réseau a appris des choses vraies de ces 10 100 lignes et fausses du
monde.

**Question 5.1.** Pourquoi une étude de régularisation a-t-elle besoin d'un modèle
nettement en overfitting ?


---

## Exercice 5.2 · Weight decay = Ridge

_Compléter, comparer · 15 min · ★★☆_

**Explication.** Ridge (séance 4) ajoute λ‖θ‖² à la loss : le gradient gagne un terme
2λθ qui tire chaque poids vers zéro. L'option `weight_decay=wd` des optimizers
PyTorch ajoute directement wd·θ au gradient : c'est la pénalité (wd/2)·‖θ‖². Vérifions
l'équivalence sur un modèle linéaire, puis mesurons son effet sur le grand réseau.

_Nuance :_ avec Adam, le terme wd·θ passe ensuite par la normalisation adaptative,
ce qui n'est plus tout à fait une pénalité L2 ; AdamW découple les deux (Loshchilov et
Hutter, 2019).


In [ ]:
# TODO : deux entraînements d'un modèle linéaire nn.Linear(77, 1), 30 epochs de SGD
# (lr=0.05, batches de 512, seed 0) :
#   (a) avec SGD(..., weight_decay=0.01) et la loss habituelle ;
#   (b) avec SGD(...) sans weight_decay, mais une loss augmentée à la main :
#       loss + (0.01 / 2) * somme des carrés de TOUS les paramètres (poids et biais).
# Comparez les poids finaux (écart maximal).
# ecart_poids = None


def lineaire(decay_optimiseur, penalite_main, wd=0.01, epoques=30):
    tr_l, _ = loaders_from_arrays(X_tr, y_tr, batch_size=512, seed=0)
    torch.manual_seed(0)
    modele = nn.Linear(X_tr.shape[1], 1)
    opt = torch.optim.SGD(
        modele.parameters(), lr=0.05, weight_decay=wd if decay_optimiseur else 0.0
    )
    lf = nn.BCEWithLogitsLoss()
    for _ in range(epoques):
        for xb, yb in tr_l:
            opt.zero_grad()
            perte = lf(modele(xb), yb)
            if penalite_main:
                # La pénalité de Ridge, écrite à la main : (wd / 2) * ||θ||².
                perte = perte + wd / 2 * sum((p**2).sum() for p in modele.parameters())
            perte.backward()
            opt.step()
    return torch.cat([p.detach().ravel() for p in modele.parameters()])


poids_a = lineaire(decay_optimiseur=True, penalite_main=False)
poids_b = lineaire(decay_optimiseur=False, penalite_main=True)
ecart_poids = (poids_a - poids_b).abs().max().item()
print(f"  norme des poids : {poids_a.norm():.4f} contre {poids_b.norm():.4f}")
print(f"  écart maximal entre les deux entraînements : {ecart_poids:.1e}")

In [ ]:
if ecart_poids is None:
    print("  à compléter")
else:
    print(f"  {'OK : même calcul' if ecart_poids < 1e-5 else 'à revoir'}")

**À vous : comparer.** Sur le grand réseau (seed 0, 40 epochs, Adam), faites varier
la force de la pénalité.


In [ ]:
# TODO : pour weight_decay dans (0, 1e-3, 1e-2, 1e-1), train_once(0, weight_decay=...)
# et la norme de tous les poids du réseau entraîné. (Pour la norme, modifiez une copie de
# train_once qui renvoie aussi le réseau, ou recalculez-la dans une petite fonction.)


def norme_apres(weight_decay):
    tr_loader, va_loader = loaders_from_arrays(
        X_tr, y_tr, X_va, y_va, batch_size=512, seed=0
    )
    torch.manual_seed(0)
    net = make_mlp(X_tr.shape[1], hidden=BIG)
    fit(
        net, tr_loader, va_loader, epochs=40, lr=1e-3, weight_decay=weight_decay, seed=0
    )
    perte_va, logits = evaluate(net, va_loader, nn.BCEWithLogitsLoss())
    perte_tr, _ = evaluate(net, train_eval_loader, nn.BCEWithLogitsLoss())
    norme = torch.cat([p.detach().ravel() for p in net.parameters()]).norm().item()
    return roc_auc_score(y_va, logits.ravel()), perte_va - perte_tr, norme


lignes = []
for wd in (0.0, 1e-3, 1e-2, 1e-1):
    auc, ecart, norme = norme_apres(wd)
    lignes.append(
        {
            "weight_decay": wd,
            "AUC": round(auc, 4),
            "gap final": round(ecart, 3),
            "norme des poids": round(norme, 1),
        }
    )
print(pd.DataFrame(lignes).to_string(index=False))

**Analyse.** Les deux calculs du modèle linéaire donnent **les mêmes poids** : weight
decay est la pénalité de Ridge. Sur le grand réseau, la pénalité rétrécit les poids et
le gap ; l'AUC s'améliore jusqu'à un point, puis une pénalité trop forte mène à
l'underfitting. C'est le cadran du bias-variance tradeoff de la séance 4.

**Question 5.2.** En séance 4, Ridge changeait le R² de 0,0002 ; ici, le weight decay
change l'AUC de plusieurs centièmes. Expliquez la différence en une phrase.


---

## Exercice 5.3 · Le dropout à la main

_Compléter · 12 min · ★★☆_

**Explication.** Pendant l'entraînement, le dropout met à zéro chaque unité avec la
probabilité p, et multiplie les survivantes par 1/(1 - p) pour que la somme attendue
ne change pas. À l'évaluation, il ne fait rien.

![Deux tirages de dropout](../assets/diagrams/s09_s10_training/nn_dropout.png)

_Schéma de la bibliothèque du cours (provenance non établie, usage en classe)._


In [ ]:
# TODO : écrivez dropout(a, p, entrainement, rng) :
#   - en entraînement : masque = rng.random(a.shape) >= p ; renvoyer a * masque / (1 - p) ;
#   - sinon : renvoyer a inchangé.


# def dropout(a, p, entrainement, rng):
#     raise NotImplementedError


def dropout(a, p, entrainement, rng):
    if not entrainement:
        return a
    masque = rng.random(a.shape) >= p  # True avec la probabilité 1 - p
    return a * masque / (1 - p)  # l'espérance est conservée

In [ ]:
# Vérifications : espérance conservée, proportion de zéros, et eval mode.
rng = np.random.default_rng(0)
a = np.array([1.0, 2.0, 3.0, 4.0])
tirages = np.array([dropout(a, 0.5, True, rng) for _ in range(20000)])
print(f"  moyenne des tirages : {tirages.mean(axis=0).round(3)}   (a = {a})")
print(f"  proportion de zéros : {(tirages == 0).mean():.3f}   (p = 0,5)")
print(f"  eval mode           : {dropout(a, 0.5, False, rng)}")

couche = nn.Dropout(0.5)
t = torch.ones(10000)
couche.train()
vals = couche(t)
print(
    f"  nn.Dropout en train mode : valeurs {sorted(set(vals.tolist()))}, zéros {(vals == 0).float().mean():.3f}"
)
couche.eval()
print(f"  nn.Dropout en eval mode  : valeurs {sorted(set(couche(t).tolist()))}")

**Analyse.** L'espérance de chaque unité est conservée, et `nn.Dropout` fait
exactement ce calcul : 0 ou 2 (= 1/(1 - 0,5)) en train mode, 1 en eval mode. Chaque
training step ajuste donc un sous-réseau différent : aucune unité ne peut compter
sur la présence d'une autre (co-adaptation). Et le comportement **dépend du mode** :
c'est pour lui que `model.eval()` existe.

**Question 5.3.** Pourquoi multiplier par 1/(1 - p) pendant l'entraînement, plutôt que
multiplier par (1 - p) pendant l'évaluation ?


---

## Exercice 5.4 · Batch norm et layer norm

_Compléter, analyser · 12 min · ★★☆_

**Explication.** Les deux normalisent une activation (moyenne 0, variance 1) puis
appliquent une échelle et un décalage appris. La différence est l'**axe** :

![Axes de normalisation](../assets/generated/cours_08_10_fr/f15_normalisation.png)

- la **batch norm** normalise chaque unité sur les exemples du batch ;
- la **layer norm** normalise chaque exemple sur les unités de la couche.


In [ ]:
# TODO : écrivez batch_norm(A) et layer_norm(A) (sans échelle ni décalage appris), avec
# la variance non corrigée (np.var) et eps = 1e-5, puis comparez-les à nn.BatchNorm1d(5)
# en train mode et à nn.LayerNorm(5) sur la matrice A ci-dessous.
# Écrivez aussi mlp_layer_norm(n_entrees, hidden) : comme make_mlp, mais avec
# nn.LayerNorm(largeur) après la nn.Linear de chaque hidden layer (il servira en 5.5).
# A = np.random.default_rng(3).normal(0, 1, (6, 5)).astype(np.float32)


# def batch_norm(A, eps=1e-5):
#     raise NotImplementedError


# def layer_norm(A, eps=1e-5):
#     raise NotImplementedError


# def mlp_layer_norm(n_entrees, hidden):
#     raise NotImplementedError


A = np.random.default_rng(3).normal(0, 1, (6, 5)).astype(np.float32)


def batch_norm(A, eps=1e-5):
    # Chaque colonne (une unité) normalisée sur les lignes (les exemples du batch).
    return (A - A.mean(axis=0)) / np.sqrt(A.var(axis=0) + eps)


def layer_norm(A, eps=1e-5):
    # Chaque ligne (un exemple) normalisée sur ses colonnes (les unités).
    return (A - A.mean(axis=1, keepdims=True)) / np.sqrt(
        A.var(axis=1, keepdims=True) + eps
    )


def mlp_layer_norm(n_entrees, hidden):
    couches, precedent = [], n_entrees
    for largeur in hidden:
        couches += [nn.Linear(precedent, largeur), nn.LayerNorm(largeur), nn.ReLU()]
        precedent = largeur
    couches.append(nn.Linear(precedent, 1))
    return nn.Sequential(*couches)

In [ ]:
bn, ln = nn.BatchNorm1d(5), nn.LayerNorm(5)
bn.train()
with torch.no_grad():
    ecart_bn = np.abs(batch_norm(A) - bn(torch.tensor(A)).numpy()).max()
    ecart_ln = np.abs(layer_norm(A) - ln(torch.tensor(A)).numpy()).max()
print(
    f"  batch_norm contre nn.BatchNorm1d : {ecart_bn:.1e}   {'OK' if ecart_bn < 1e-5 else 'à revoir'}"
)
print(
    f"  layer_norm contre nn.LayerNorm   : {ecart_ln:.1e}   {'OK' if ecart_ln < 1e-5 else 'à revoir'}"
)

# La dépendance au batch : le même exemple (la ligne 0), accompagné de voisins différents.
autre_lot = np.vstack(
    [A[:1], np.random.default_rng(9).normal(2, 3, (5, 5)).astype(np.float32)]
)
print(f"\n  ligne 0, batch norm, batch 1 : {batch_norm(A)[0].round(3)}")
print(f"  ligne 0, batch norm, batch 2 : {batch_norm(autre_lot)[0].round(3)}")
print(f"  ligne 0, layer norm, batch 1 : {layer_norm(A)[0].round(3)}")
print(f"  ligne 0, layer norm, batch 2 : {layer_norm(autre_lot)[0].round(3)}")

**Analyse.** Avec la batch norm, la sortie d'un exemple **dépend de ses voisins de
batch** : c'est ce qui la rend sensible à la batch size, et ce qui oblige à
utiliser, en eval mode, des running statistics accumulées pendant
l'entraînement (la seconde chose que change `model.eval()`). La layer norm ne regarde
qu'un exemple à la fois : même calcul en train mode et en eval mode, et valable
pour un batch d'un seul exemple.

**Question 5.4.** Pourquoi la layer norm est-elle préférée dans les transformers,
qui traitent des séquences de longueurs variables, souvent par petits batches ?


---

## Exercice 5.5 · L'ablation

_Résoudre · 28 min · ★★★_

**Explication.** Une **ablation** mesure l'effet de chaque traitement en ne changeant
**qu'un facteur à la fois**, tout le reste fixé, et en **répétant avec plusieurs
seeds**, parce qu'un effet plus petit que la variation d'une seed à l'autre n'est
pas un effet. On reprend les six configurations de la séance 10 et on en ajoute deux :

- **+ layer norm** (exercice 5.4) ;
- **+ bruit gaussien 0,3** : une **data augmentation**. À chaque batch, on ajoute un
  bruit N(0 ; 0,3²) aux 11 features numériques standardisées. Pour une image, on
  retournerait ou recadrerait ; pour un tableau, il n'existe pas d'invariance aussi
  évidente, et le bruit est l'augmentation la plus simple.

L'augmentation vit **dans le chargeur de données** : c'est lui qui fabrique les batches.


In [ ]:
def chargeur_bruite(sigma, batch_size, graine):
    """Chargeur d'entraînement qui bruite les colonnes numériques de chaque batch."""
    generateur_bruit = torch.Generator().manual_seed(graine + 1000)

    def assembler(lot):
        xb = torch.stack([x for x, _ in lot])
        yb = torch.stack([t for _, t in lot])
        xb[:, :N_NUM] += sigma * torch.randn(len(xb), N_NUM, generator=generateur_bruit)
        return xb, yb

    return DataLoader(
        TabularDataset(X_tr, y_tr),
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(graine),
        collate_fn=assembler,
    )


def entrainer(
    graine,
    *,
    dropout=0.0,
    weight_decay=0.0,
    norm=None,
    arret=False,
    bruit=0.0,
    epoques=40,
    batch_size=512,
):
    """train_once étendu : layer norm et bruit en plus."""
    tr_loader, va_loader = loaders_from_arrays(
        X_tr, y_tr, X_va, y_va, batch_size=batch_size, seed=graine
    )
    if bruit > 0:
        tr_loader = chargeur_bruite(bruit, batch_size, graine)
    torch.manual_seed(graine)
    if norm == "layer":
        net = mlp_layer_norm(X_tr.shape[1], BIG)
    else:
        net = make_mlp(
            X_tr.shape[1], hidden=BIG, dropout=dropout, batch_norm=(norm == "batch")
        )
    stopper = EarlyStopping(patience=5) if arret else None
    h = fit(
        net,
        tr_loader,
        va_loader,
        epochs=epoques,
        lr=1e-3,
        weight_decay=weight_decay,
        early_stopping=stopper,
        seed=graine,
    )
    lf = nn.BCEWithLogitsLoss()
    perte_va, logits = evaluate(net, va_loader, lf)
    # Le gap comme dans train_once : modèle final, eval mode, et training set
    # sans bruit.
    perte_tr, _ = evaluate(net, train_eval_loader, lf)
    return {
        "AUC": roc_auc_score(y_va, logits.ravel()),
        "epochs": len(h["train"]),
        "gap": perte_va - perte_tr,
        "val loss": perte_va,
        "historique": h,
    }

**À vous (1) : avant d'exécuter**, écrivez votre classement et votre règle de
décision. Une règle fixée après avoir vu les résultats n'est plus une règle.


In [ ]:
# TODO : vos prédictions et votre règle, AVANT d'exécuter l'ablation.
mon_classement = "____"  # du plus utile au moins utile
sans_effet_selon_moi = "____"  # lesquels n'aideront pas ?
regle = "un effet est réel si |ΔAUC| > 2 x l'écart-type entre seeds de la baseline"

**À vous (2) : l'ablation.**


In [ ]:
# TODO : définissez CONFIGS (les six de la séance 10, puis "+ layer norm" et
# "+ bruit 0,3"), GRAINES = (0, 1, 2), et pour chaque configuration :
# AUC moyenne, écart-type (ddof=1), AUC par seed, epochs, gap moyen, validation loss
# moyenne. Rangez le tout dans un DataFrame `ablation`.
# Séance 10 : "baseline (none)" {}, "+ early stopping" (arret=True),
# "+ weight decay 1e-2", "+ dropout 0.5", "+ batch norm" (norm="batch"),
# "all combined" (arret, weight_decay=1e-3, dropout=0.2, norm="batch").
# ablation = None

CONFIGS = {
    "baseline (none)": {},
    "+ early stopping": dict(arret=True),
    "+ weight decay 1e-2": dict(weight_decay=1e-2),
    "+ dropout 0.5": dict(dropout=0.5),
    "+ batch norm": dict(norm="batch"),
    "all combined": dict(arret=True, weight_decay=1e-3, dropout=0.2, norm="batch"),
    "+ layer norm": dict(norm="layer"),
    "+ bruit 0,3": dict(bruit=0.3),
}
GRAINES = (0, 1, 2)  # (0,) pour un passage rapide, mais sans estimation du bruit

debut = time.monotonic()
lignes, courbes = [], {}
for nom, reglages in CONFIGS.items():
    runs = [entrainer(g, **reglages) for g in GRAINES]
    courbes[nom] = runs[0]["historique"]
    aucs = np.array([r["AUC"] for r in runs])
    lignes.append(
        {
            "configuration": nom,
            "AUC": round(aucs.mean(), 4),
            "sd": round(aucs.std(ddof=1), 4),
            "par seed": np.round(aucs, 4).tolist(),
            "epochs": [r["epochs"] for r in runs],
            "gap": round(float(np.mean([r["gap"] for r in runs])), 3),
            "val loss": round(float(np.mean([r["val loss"] for r in runs])), 3),
        }
    )
ablation = pd.DataFrame(lignes)
print(ablation.to_string(index=False))
print(
    f"\n  ({time.monotonic() - debut:.0f} s pour {len(CONFIGS) * len(GRAINES)} entraînements)"
)

In [ ]:
# La règle de décision, fixée avant les mesures.
if ablation is None:
    print("  à compléter")
else:
    base = ablation.loc[ablation["configuration"] == "baseline (none)"].iloc[0]
    seuil = 2 * base["sd"]
    print(
        f"  écart-type de la baseline : {base['sd']:.4f}   seuil (2 écarts-types) : {seuil:.4f}\n"
    )
    for _, ligne in ablation.iterrows():
        if ligne["configuration"].startswith("baseline"):
            continue
        delta = ligne["AUC"] - base["AUC"]
        verdict = "réel" if abs(delta) > seuil else "dans le bruit des seeds"
        print(f"  {ligne['configuration']:22s} {delta:+.4f}   {verdict}")

    # Vérification : les six lignes de la séance 10.
    reference = {
        "baseline (none)": 0.8023,
        "+ early stopping": 0.8478,
        "+ weight decay 1e-2": 0.8547,
        "+ dropout 0.5": 0.8426,
        "+ batch norm": 0.7881,
        "all combined": 0.8532,
    }
    identiques = all(
        abs(ablation.set_index("configuration").loc[c, "AUC"] - v) < 5e-4
        for c, v in reference.items()
    )
    print(
        f"\n  six premières lignes identiques à la séance 10 : {'OK' if identiques else 'différent'}"
    )

In [ ]:
if ablation is not None:
    ordre = ablation.sort_values("AUC")
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(
        ordre["configuration"],
        ordre["AUC"],
        xerr=ordre["sd"],
        color="steelblue",
        capsize=3,
        zorder=2,
    )
    ax.axvspan(
        base["AUC"] - seuil,
        base["AUC"] + seuil,
        color="orange",
        alpha=0.2,
        zorder=0,
        label="baseline ± 2 écarts-types",
    )
    ax.set(
        xlim=(0.75, 0.88),
        xlabel="AUC de validation (moyenne sur 3 seeds)",
        title="Huit configurations, un facteur à la fois",
    )
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

**Analyse : les six premières lignes.** Ce sont exactement les résultats de la séance
10 : early stopping, weight decay et dropout franchissent le seuil ; la batch norm est
la seule à ne pas le franchir (point estimé négatif, plus grande variance entre
seeds, gap le plus large) ; « tout combiné » ne bat pas le weight decay seul.
Rappel de précision : « la batch norm **n'a pas aidé** », et non « elle a nui ».

**Analyse : les deux lignes nouvelles.**

- **Layer norm** : aucun effet mesurable, ni sur l'AUC ni sur le gap. Elle normalise
  des activations dans un réseau que rien n'empêchait d'apprendre, sur des entrées
  déjà standardisées.
- **Bruit 0,3** : l'AUC ne bouge pas au-delà du bruit des seeds, et pourtant
  le **gap baisse de 40 %** (0,549 contre 0,911).
  Regardez la colonne « val loss » : c'est la validation loss qui a baissé. Le
  bruit rend le réseau **moins sûr de lui** ; la cross-entropy, qui punit durement
  les erreurs commises avec assurance, s'améliore, mais l'**ordre** des annonces, seul
  mesuré par l'AUC, ne change pas.

> **Réduire le gap n'est pas améliorer le modèle.** Le gap est mesuré en loss, la
> décision se prend sur la métrique du problème (ici l'AUC). Un traitement peut corriger
> la confiance sans améliorer le classement.

Remarquez enfin comment le gap est mesuré : sur le modèle final, en eval mode.
Pour les lignes avec early stopping, ce modèle est celui de la **meilleure** epoch,
restauré à la fin ; les losses de la **dernière** epoch, celles de l'historique,
décriraient un modèle abandonné. Calculez-les pour « + early stopping » à partir de
`courbes`, et comparez.

**Question 5.5.** La configuration « all combined » contient la batch norm. Quelle
expérience permettrait de savoir si c'est elle qui l'empêche de battre le weight decay
seul ? (Vous la mènerez au TD 6.)


---

## Exercice 5.6 · Le bug silencieux

_Analyser · 5 min · ★☆☆_

**Explication.** Un réseau avec dropout et batch norm, entraîné une fois, évalué deux
fois : en eval mode (correct) et en train mode (le bug).


In [ ]:
tr_loader, va_loader = loaders_from_arrays(
    X_tr, y_tr, X_va, y_va, batch_size=512, seed=0
)
torch.manual_seed(0)
net = make_mlp(X_tr.shape[1], hidden=BIG, dropout=0.5, batch_norm=True)
fit(net, tr_loader, va_loader, epochs=40, lr=1e-3, seed=0)


def auc_mode(mode_eval):
    net.eval() if mode_eval else net.train()
    with torch.no_grad():
        sorties = np.vstack([net(xb).numpy() for xb, _ in va_loader])
    return roc_auc_score(y_va, sorties.ravel())


correct = auc_mode(True)
bug_1, bug_2 = auc_mode(False), auc_mode(False)
print(f"  avec model.eval()     : AUC {correct:.4f}")
print(
    f"  sans model.eval()     : AUC {bug_1:.4f}  puis {bug_2:.4f} (même modèle, même données)"
)
print(f"  coût du bug           : {correct - bug_1:+.4f}")

**Analyse.** 0,028 d'AUC perdus par une ligne oubliée, sans erreur ni avertissement :
trois fois l'écart entre le MLP et le booster établi au TD 4. Remarquez la seconde
mesure : **sans `model.eval()`, deux évaluations du même modèle ne donnent pas la même
AUC**. C'est le test le plus simple pour détecter ce bug : évaluer deux fois.

**Question 5.6.** Pourquoi appeler `model.eval()` même quand le modèle ne contient ni
dropout ni batch norm ?


---

## Synthèse

| technique                 | problème traité                                                | coût                                           | ce que l'ablation a montré ici                                  |
| ------------------------- | -------------------------------------------------------------- | ---------------------------------------------- | --------------------------------------------------------------- |
| early stopping            | l'entraînement continue après le minimum de la validation loss | un validation set, une patience                | fort gain, entraînement 4 fois plus court                       |
| weight decay              | des poids qui grandissent sans contrainte                      | une force à régler                             | le meilleur traitement seul ; c'est Ridge                       |
| dropout                   | la co-adaptation des unités                                    | convergence plus lente, asymétrie train / eval | aide, sans « corriger » l'overfitting                           |
| batch norm                | des activations mal échelonnées                                | dépend du batch, asymétrie train / eval        | n'a pas aidé (pas de nuisance démontrée)                        |
| layer norm                | idem, un exemple à la fois                                     | peu de chose                                   | aucun effet mesurable                                           |
| data augmentation (bruit) | trop peu de diversité dans les données                         | un niveau de bruit à choisir                   | aucun gain d'AUC ; gap réduit de 40 % (modèle moins sûr de lui) |

- Un effet plus petit que la variation entre seeds **n'est pas un effet**.
- Mesurer les régularisateurs **un par un**, ne combiner que ce qui a mérité sa place.
- Comparer des gaps **mesurés dans les mêmes conditions**, et décider sur la
  validation.
- `model.eval()`, toujours.

## Questions de compréhension

1. Montrez que `weight_decay=wd` correspond à la pénalité (wd/2)·‖θ‖² avec SGD.
2. Pourquoi le dropout multiplie-t-il les survivants par 1/(1 - p) ?
3. Quelle est la différence d'axe entre batch norm et layer norm, et sa conséquence ?
4. Pourquoi l'ablation utilise-t-elle plusieurs seeds sur un fold fixe ?
5. Le bruit réduit le gap de 40 % sans changer l'AUC. Comment est-ce possible ?

## Pour aller plus loin

- Srivastava, N. et al. (2014), _Dropout: A Simple Way to Prevent Neural Networks from
  Overfitting_, JMLR 15. https://jmlr.org/papers/v15/srivastava14a.html
- Ioffe, S. et Szegedy, C. (2015), _Batch Normalization_. https://arxiv.org/abs/1502.03167
- Santurkar, S. et al. (2018), _How Does Batch Normalization Help Optimization?_
  https://arxiv.org/abs/1805.11604
- Ba, J., Kiros, J. et Hinton, G. (2016), _Layer Normalization_.
  https://arxiv.org/abs/1607.06450
- Bishop, C. (1995), _Training with noise is equivalent to Tikhonov regularization_,
  Neural Computation 7(1), 108-116.
- Shorten, C. et Khoshgoftaar, T. (2019), _A survey on Image Data Augmentation for Deep
  Learning_, Journal of Big Data 6, 60.
- Loshchilov, I. et Hutter, F. (2019), _Decoupled Weight Decay Regularization_ (AdamW).
  https://arxiv.org/abs/1711.05101
- Goodfellow, Bengio et Courville (2016), _Deep Learning_, chapitre 7.

**Prochain TD :** les mini-défis, en autonomie.
